In [3]:
import pandas as pd 
from sklearn.model_selection import train_test_split
import numpy as np


In [4]:
# SETUP 

TRAINING_DATA = "training_data_clean.csv"


# def prep_data():
df = pd.read_csv(TRAINING_DATA)

# rename
df.columns = [
    'id', 
    'best_tasks_free', 
    'acad_tasks_rating', 
    'best_tasks_select', 
    'subopt_freq_rating',  
    'subopt_tasks_select', 
    'subopt_tasks_free', 
    'evidence_freq_rating', 
    'verify_freq_rating', 
    'verify_method_free', 
    'target'
    ]

for task in df['best_tasks_select'].unique():
    print(task)

# prep_data()


Math computations,Data processing or analysis,Explaining complex concepts simply,Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés)
Writing or debugging code
Math computations,Converting content between formats (e.g., LaTeX)
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
Math computations,Writing or debugging code,Data processing or analysis,Converting content between formats (e.g., LaTeX)
Brainstorming or generating creative ideas
Writing or debugging code,Data processing or analysis,Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Writing or editing essays/reports,Drafting professional text (e.g., emails, résumés),Brainstorming or generating creative ideas
nan
Explaining complex concepts simply,Converting content between formats (e.g., LaTeX),Brainstormi

In [5]:
# EXPLORE DATA 

df.isnull().sum()

id                       0
best_tasks_free          2
acad_tasks_rating        0
best_tasks_select       15
subopt_freq_rating       1
subopt_tasks_select     22
subopt_tasks_free       11
evidence_freq_rating    62
verify_freq_rating       4
verify_method_free      10
target                   0
dtype: int64

In [5]:
import numpy as np
from sklearn.compose import ColumnTransformer
 # TODO: can use KNN to find the imputation point instead, in-person imputation 
from sklearn.impute import SimpleImputer
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

# divide columns by type 
text_cols = [
    'best_tasks_free', 
    'subopt_tasks_free', 
    'verify_method_free',
    ] 


ord_cols = [
    'acad_tasks_rating', 
    'subopt_freq_rating',  
    'evidence_freq_rating', 
    'verify_freq_rating', 
    ]

ord_categories = [
                '1 — Not at all likely',
                '2 — Unlikely',
                '3 — Neutral / Unsure',
                '4 — Likely',
                '5 — Very likely'
                ]

cat_cols = [
    'best_tasks_select', 
    'subopt_tasks_select',
    ]
    
def preprocess_text():
    return make_pipeline(
        SimpleImputer(strategy="constant", fill_value ="", missing_values=np.nan)
        # text embedding -> always use llm, tokenize and get embedding vector, using hugging face library (Ex. BERT)
        # return torch tensor, embedding is one of the properties 
        
        # TFIDF manually (classical)
    )
    
def preprocess_ord(): 
    return make_pipeline(
        SimpleImputer(strategy="most_frequent"),
        OrdinalEncoder(categories= [ord_categories] * len(ord_cols)),
    # TODO: encode the dimension of the overall feature vector, do KNN on that, euclidian disatnce between datapoints  
    # SimpleImputer(strategy="constant", fill_value="most_frequent", missing_values=np.nan),
    )
    
def preprocess_cat():
    pass 
    # TODO: research how to preprocess these
    
def create_preprocessor(df):
    # create pipelines for each type, applied per column within each subset
    # pipelines run in tandem 
    transformers = [
        ("preprocessing_pipeline_for_free_response_features", preprocess_text(), text_cols),
        ("preprocessing_pipeline_for_ordinal_rating_features", preprocess_ord(), ord_cols),
         ("preprocessing_pipeline_for_categorical_select_features", preprocess_cat(), cat_cols),
    ]
    
    return ColumnTransformer(transformers=transformers)


# column_transformer = create_preprocessor()
# column_transformer.set_output(transform="pandas")
# preprocessed_num_cat_features_df = column_transformer.fit_transform(
#     train_df[[NUMERICAL_FEATURE, CATEGORICAL_FEATURE]]
# )

In [ ]:
# LLM EMBEDDINGS

# LLM EMBEDDINGS

import torch
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
# use pretrained weights

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def embed_texts(text_col, batch_size=32):
    """
    Accepts a pandas Series, list, numpy array, or single string and returns
    a numpy array of mean-pooled embeddings (N, hidden_size).
    Replaces missing values with empty strings so the tokenizer always gets str inputs.
    """
    # normalize input to a list of strings
    texts = [ '' if pd.isna(x) else str(x) for x in text_col ]

    if len(texts) == 0:
        return np.zeros((0, model.config.hidden_size))

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        # pass to model with no gradient bc we are not training, just getting embeddings
        with torch.no_grad():
            # done per response in batch  ============================================
            outputs = model(**enc) # outputs.last_hidden_state: (B, seq_len, 768)

            # Mean Pooling: average vectors of each true token vector in the text ====
            mask = enc["attention_mask"].unsqueeze(-1) # (B, seq_len, 1), add 1 dim for multiplication
            masked = outputs.last_hidden_state * mask # ignores padding tokens
            summed = masked.sum(dim=1) 
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts  # (B, hidden)

            all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

embed_best_tasks_free = embed_texts(df['best_tasks_free'])
print(embed_best_tasks_free)
# embed_subopt_tasks_free = embed_texts(df['subopt_tasks_free'])
# embed_verify_method_free = embed_texts(df['verify_method_free'])


# def embed_texts(text_cols, df, batch_size=32):
#     col_embedding = []
#     for col in text_cols:
#         embed_texts(df[col], batch_size=batch_size)
#         col_embedding.append(col)
#     return np.hstack(col_embedding)
     
# print(embed_texts(text_cols, df))
from transformers import AutoTokenizer, AutoModel
import numpy as np
import pandas as pd
# use pretrained weights

MODEL_NAME = "distilbert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model.eval()

def embed_texts(text_col, batch_size=32):
    """
    Accepts a pandas Series, list, numpy array, or single string and returns
    a numpy array of mean-pooled embeddings (N, hidden_size).
    Replaces missing values with empty strings so the tokenizer always gets str inputs.
    """
    # normalize input to a list of strings
    texts = [ '' if pd.isna(x) else str(x) for x in text_col ]

    if len(texts) == 0:
        return np.zeros((0, model.config.hidden_size))

    all_embeddings = []

    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]

        enc = tokenizer(
            batch,
            padding=True,
            truncation=True,
            return_tensors="pt"
        )

        # pass to model with no gradient bc we are not training, just getting embeddings
        with torch.no_grad():
            # done per response in batch  ============================================
            outputs = model(**enc) # outputs.last_hidden_state: (B, seq_len, 768)

            # Mean Pooling: average vectors of each true token vector in the text ====
            mask = enc["attention_mask"].unsqueeze(-1) # (B, seq_len, 1), add 1 dim for multiplication
            masked = outputs.last_hidden_state * mask # ignores padding tokens
            summed = masked.sum(dim=1) 
            counts = mask.sum(dim=1).clamp(min=1e-9)
            mean_pooled = summed / counts  # (B, hidden)

            all_embeddings.append(mean_pooled.cpu().numpy())

    return np.vstack(all_embeddings)

embed_best_tasks_free = embed_texts(df['best_tasks_free'])
print(embed_best_tasks_free)
# embed_subopt_tasks_free = embed_texts(df['subopt_tasks_free'])
# embed_verify_method_free = embed_texts(df['verify_method_free'])


# def embed_texts(text_cols, df, batch_size=32):
#     col_embedding = []
#     for col in text_cols:
#         embed_texts(df[col], batch_size=batch_size)
#         col_embedding.append(col)
#     return np.hstack(col_embedding)
     
# print(embed_texts(text_cols, df))

AttributeError: partially initialized module 'torch' has no attribute 'types' (most likely due to a circular import)

In [ ]:

def split_data(df):
    students = df['id'].unique()
    train_df, test_df = train_test_split(students, test_size=0.3, random_state=42)
